# Fine-tune both classifiers for AI-generated vs real images

Run this notebook in Google Colab, preferably with a GPU (**Runtime → Change runtime type → T4 GPU**). It fine-tunes the existing CIFAKE Custom CNN checkpoint and the pinned Hugging Face ViT on Kaggle's [AI Generated Images vs Real Images](https://www.kaggle.com/datasets/cashbowman/ai-generated-images-vs-real-images) dataset, which includes varied subjects such as people, animals, portraits, scenery, and artwork. KaggleHub downloads about 500 MB. It does **not** train from scratch.

The notebook makes one shared stratified train/validation/test split. Validation guides checkpoint selection; the test split is evaluated only after training. The CNN keeps its app preprocessing (RGB bilinear resize to 32×32, pixels in [0, 1]); the ViT uses its pinned Hugging Face image processor. Labels are REAL=0 and AI-generated=1.

**Important:** the models are optimized for AI-vs-real classification, not image-edit/tampering detection. Keep the original checkpoints. New checkpoints and metrics are written separately to `ai_vs_real_finetuning_outputs`; the app will not switch to them automatically. This dataset's Kaggle license is the Community Data License Agreement - Sharing v1.0; review its terms before redistributing data or model outputs. A random held-out split from one dataset does not prove performance on unseen generators or real-world images.

## 1. Install packages and select a device

In [ ]:
%pip -q install kagglehub "transformers>=4.39.3,<5" scikit-learn pandas pillow safetensors

import gc
import json
import os
import re
import random
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import kagglehub
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from transformers import AutoImageProcessor, AutoModelForImageClassification

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"TensorFlow: {tf.__version__}; PyTorch: {torch.__version__}")
print(f"TensorFlow GPU(s): {tf.config.list_physical_devices('GPU')}")
print(f"ViT device: {DEVICE}")

## 2. Download the Kaggle dataset and locate the existing CNN checkpoint

Kaggle may request account authentication. The dataset downloads using the KaggleHub call below. If your trained Custom CNN checkpoint is already in Colab, the notebook searches `/content` for `cifake_custom_cnn.keras` (including a `cifake/` subfolder); otherwise upload that one checkpoint when prompted.

In [ ]:
dataset_path = Path(kagglehub.dataset_download("cashbowman/ai-generated-images-vs-real-images"))
print("Path to dataset files:", dataset_path)
if not dataset_path.is_dir():
    raise FileNotFoundError(f"kagglehub did not return a dataset directory: {dataset_path}")

from google.colab import files

checkpoint_candidates = sorted(Path("/content").rglob("cifake_custom_cnn.keras"))
if len(checkpoint_candidates) == 1:
    CNN_CHECKPOINT = checkpoint_candidates[0]
elif len(checkpoint_candidates) > 1:
    raise ValueError(f"Found multiple cifake_custom_cnn.keras checkpoints; set CNN_CHECKPOINT explicitly: {checkpoint_candidates}")
else:
    print("Upload only the CIFAKE Custom CNN checkpoint named cifake_custom_cnn.keras.")
    uploaded = files.upload()
    checkpoint_names = [name for name in uploaded if Path(name).name == "cifake_custom_cnn.keras"]
    if len(checkpoint_names) != 1:
        raise ValueError(f"Expected exactly cifake_custom_cnn.keras; received {list(uploaded)}")
    CNN_CHECKPOINT = Path("/content") / checkpoint_names[0]
if not CNN_CHECKPOINT.is_file():
    raise FileNotFoundError(f"CNN checkpoint not found: {CNN_CHECKPOINT}")
print(f"Starting CNN checkpoint: {CNN_CHECKPOINT}")

## 3. Inspect dataset labels and create one shared split

Images are labeled from explicit class-directory names only, not from filenames. It recognizes common folder labels such as `Real Images`, `Real`, `AI Images`, `AI Generated Images`, `Fake`, and `Generated`. It prints class counts and fails if both classes are not confidently identified.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
AI_FOLDER_NAMES = {
    "ai", "ai image", "ai images", "ai generated", "ai generated image",
    "ai generated images", "generated", "generated image", "generated images",
    "fake", "fake image", "fake images", "synthetic", "synthetic images",
    "1 fake", "1 ai", "ai generated content",
}
REAL_FOLDER_NAMES = {
    "real", "real image", "real images", "real photo", "real photos",
    "authentic", "authentic images", "0 real", "real world",
}

records = []
unlabeled_images = []
for image_path in sorted(dataset_path.rglob("*")):
    if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue
    relative_parent = image_path.parent.relative_to(dataset_path)
    label = None
    for folder in reversed(relative_parent.parts):
        normalized = re.sub(r"[^a-z0-9]+", " ", folder.lower()).strip()
        if normalized in AI_FOLDER_NAMES:
            label = 1
            break
        if normalized in REAL_FOLDER_NAMES:
            label = 0
            break
    if label is None:
        unlabeled_images.append(str(image_path))
        continue
    records.append({"path": str(image_path), "label": label})

frame = pd.DataFrame(records).drop_duplicates(subset="path").reset_index(drop=True)
if frame.empty or set(frame["label"].unique()) != {0, 1}:
    raise ValueError(
        "Could not identify both real and AI image folders. Inspect the downloaded "
        f"folder names under {dataset_path}. Unlabeled image examples: {unlabeled_images[:10]}"
    )
class_counts = frame["label"].value_counts().sort_index().to_dict()
print(f"Discovered {len(frame):,} images; REAL={class_counts[0]:,}, AI={class_counts[1]:,}")
print(f"Skipped images outside recognized class folders: {len(unlabeled_images):,}")
if min(class_counts.values()) < 10:
    raise ValueError(f"Too few images in a class for a reliable split: {class_counts}")

train_frame, remainder = train_test_split(
    frame, test_size=0.30, random_state=SEED, stratify=frame["label"]
)
validation_frame, test_frame = train_test_split(
    remainder, test_size=0.50, random_state=SEED, stratify=remainder["label"]
)
train_frame = train_frame.reset_index(drop=True)
validation_frame = validation_frame.reset_index(drop=True)
test_frame = test_frame.reset_index(drop=True)
for split_name, split_frame in (("train", train_frame), ("validation", validation_frame), ("test", test_frame)):
    print(f"{split_name}: {len(split_frame):,} images; counts={split_frame['label'].value_counts().sort_index().to_dict()}")
assert not (set(train_frame.path) & set(validation_frame.path))
assert not (set(train_frame.path) & set(test_frame.path))
assert not (set(validation_frame.path) & set(test_frame.path))

## 4. Fine-tune the existing CIFAKE Custom CNN

The existing CNN is a 32×32 binary classifier. This continues its weights with a lower learning rate, using the same input scaling as the app. Validation loss selects the checkpoint; the test split is not used for training or checkpoint selection.

In [ ]:
OUTPUT_DIR = Path("/content/ai_vs_real_finetuning_outputs")
CNN_OUTPUT_DIR = OUTPUT_DIR / "cnn"
VIT_OUTPUT_DIR = OUTPUT_DIR / "huggingface_vit"
METRICS_DIR = OUTPUT_DIR / "metrics"
for directory in (CNN_OUTPUT_DIR, VIT_OUTPUT_DIR, METRICS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CNN_SIZE = (32, 32)
CNN_BATCH_SIZE = 64
CNN_EPOCHS = 8
CNN_LEARNING_RATE = 1e-5
AUTOTUNE = tf.data.AUTOTUNE

def decode_cnn_image(path_tensor):
    def load_one(path_value):
        path = path_value.decode("utf-8")
        try:
            with Image.open(path) as image:
                image = image.convert("RGB").resize(CNN_SIZE, Image.Resampling.BILINEAR)
                return np.asarray(image, dtype=np.uint8)
        except (OSError, UnidentifiedImageError) as error:
            raise RuntimeError(f"Cannot decode image {path}: {error}") from error
    tensor = tf.numpy_function(load_one, [path_tensor], tf.uint8)
    tensor.set_shape((32, 32, 3))
    return tf.cast(tensor, tf.float32) / 255.0

def make_cnn_dataset(split_frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((
        split_frame["path"].astype(str).to_numpy(),
        split_frame["label"].to_numpy(dtype=np.float32),
    ))
    dataset = dataset.map(
        lambda path, label: (decode_cnn_image(path), label),
        num_parallel_calls=AUTOTUNE,
    )
    if training:
        dataset = dataset.shuffle(len(split_frame), seed=SEED, reshuffle_each_iteration=True)
    return dataset.batch(CNN_BATCH_SIZE).prefetch(AUTOTUNE)

cnn_model = tf.keras.models.load_model(CNN_CHECKPOINT, compile=False)
if tuple(cnn_model.input_shape[1:]) != (32, 32, 3):
    raise ValueError(f"Expected a CIFAKE (32,32,3) CNN checkpoint; got {cnn_model.input_shape}")
if cnn_model.output_shape[-1] != 1:
    raise ValueError(f"Expected a single sigmoid output; got {cnn_model.output_shape}")
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CNN_LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")],
)
cnn_best_path = CNN_OUTPUT_DIR / "cifake_custom_cnn_ai_vs_real_best.keras"
cnn_history = cnn_model.fit(
    make_cnn_dataset(train_frame, training=True),
    validation_data=make_cnn_dataset(validation_frame),
    epochs=CNN_EPOCHS,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            cnn_best_path, monitor="val_loss", mode="min", save_best_only=True
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", mode="min", patience=2, restore_best_weights=True
        ),
    ],
    verbose=1,
)
cnn_model = tf.keras.models.load_model(cnn_best_path, compile=False)
cnn_scores = np.concatenate([
    np.asarray(cnn_model(images, training=False)).reshape(-1)
    for images, _ in make_cnn_dataset(test_frame)
])
cnn_true = test_frame["label"].to_numpy(dtype=np.int64)
cnn_model.save(CNN_OUTPUT_DIR / "cifake_custom_cnn_ai_vs_real_finetuned.keras")
print(f"Saved fine-tuned CNN: {CNN_OUTPUT_DIR / 'cifake_custom_cnn_ai_vs_real_finetuned.keras'}")
del cnn_model
tf.keras.backend.clear_session()
gc.collect()

## 5. Fine-tune the Hugging Face ViT

The ViT starts from the same pinned model used in the app. Training uses its own image processor and preserves its class-index mapping; the AI score for reporting is always the checkpoint's `FAKE` softmax probability.

In [ ]:
MODEL_ID = "dima806/ai_vs_real_image_detection"
MODEL_REVISION = "5d647683672ffd0080441f4e8b4345934c44cc61"
VIT_BATCH_SIZE = 16
VIT_EPOCHS = 5
VIT_LEARNING_RATE = 2e-5

processor = AutoImageProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
vit_model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, use_safetensors=True
)
id2label = {
    int(index): str(label).strip().upper()
    for index, label in vit_model.config.id2label.items()
}
real_ids = [index for index, label in id2label.items() if label == "REAL"]
fake_ids = [index for index, label in id2label.items() if label == "FAKE"]
if len(real_ids) != 1 or len(fake_ids) != 1:
    raise ValueError(f"Expected one REAL and one FAKE output class; got {id2label}")
REAL_CLASS_ID, FAKE_CLASS_ID = real_ids[0], fake_ids[0]
label2class_id = {0: REAL_CLASS_ID, 1: FAKE_CLASS_ID}
print(f"ViT labels: {id2label}; training target mapping: {label2class_id}")

class AIRealImageDataset(torch.utils.data.Dataset):
    def __init__(self, split_frame):
        self.paths = split_frame["path"].tolist()
        self.labels = split_frame["label"].to_numpy(dtype=np.int64)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image_path = self.paths[index]
        try:
            with Image.open(image_path) as image:
                inputs = processor(images=image.convert("RGB"), return_tensors="pt")
        except (OSError, UnidentifiedImageError) as error:
            raise RuntimeError(f"Cannot decode image {image_path}: {error}") from error
        label = label2class_id[int(self.labels[index])]
        return inputs["pixel_values"].squeeze(0), label

def collate_face_batch(batch):
    pixel_values, labels = zip(*batch)
    return torch.stack(pixel_values), torch.tensor(labels, dtype=torch.long)

train_loader = torch.utils.data.DataLoader(
    AIRealImageDataset(train_frame), batch_size=VIT_BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=torch.cuda.is_available(), collate_fn=collate_face_batch,
)
validation_loader = torch.utils.data.DataLoader(
    AIRealImageDataset(validation_frame), batch_size=VIT_BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available(), collate_fn=collate_face_batch,
)
test_loader = torch.utils.data.DataLoader(
    AIRealImageDataset(test_frame), batch_size=VIT_BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available(), collate_fn=collate_face_batch,
)

vit_model.to(DEVICE)
optimizer = torch.optim.AdamW(vit_model.parameters(), lr=VIT_LEARNING_RATE)
best_val_accuracy = -1.0
epochs_without_improvement = 0
vit_history = []
for epoch in range(VIT_EPOCHS):
    vit_model.train()
    train_loss_sum = 0.0
    train_count = 0
    for pixel_values, class_ids in train_loader:
        pixel_values = pixel_values.to(DEVICE, non_blocking=True)
        class_ids = class_ids.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        output = vit_model(pixel_values=pixel_values, labels=class_ids)
        output.loss.backward()
        optimizer.step()
        train_loss_sum += float(output.loss.item()) * len(class_ids)
        train_count += len(class_ids)

    vit_model.eval()
    validation_true, validation_scores = [], []
    with torch.inference_mode():
        for pixel_values, class_ids in validation_loader:
            logits = vit_model(pixel_values=pixel_values.to(DEVICE, non_blocking=True)).logits
            fake_probabilities = torch.softmax(logits, dim=-1)[:, FAKE_CLASS_ID]
            semantic_labels = [0 if int(value) == REAL_CLASS_ID else 1 for value in class_ids]
            validation_true.extend(semantic_labels)
            validation_scores.extend(fake_probabilities.cpu().tolist())
    val_predictions = (np.asarray(validation_scores) >= 0.5).astype(np.int64)
    val_accuracy = float(accuracy_score(validation_true, val_predictions))
    epoch_record = {
        "epoch": epoch + 1,
        "train_loss": train_loss_sum / train_count,
        "validation_accuracy": val_accuracy,
    }
    vit_history.append(epoch_record)
    print(epoch_record)
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_without_improvement = 0
        vit_model.save_pretrained(VIT_OUTPUT_DIR, safe_serialization=True)
        processor.save_pretrained(VIT_OUTPUT_DIR)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= 2:
            print("Early stopping: validation accuracy did not improve.")
            break

with (METRICS_DIR / "huggingface_vit_training_history.json").open("w", encoding="utf-8") as file:
    json.dump(vit_history, file, indent=2)
vit_model = AutoModelForImageClassification.from_pretrained(VIT_OUTPUT_DIR).to(DEVICE).eval()
vit_test_true, vit_test_scores = [], []
with torch.inference_mode():
    for pixel_values, class_ids in test_loader:
        logits = vit_model(pixel_values=pixel_values.to(DEVICE, non_blocking=True)).logits
        fake_probabilities = torch.softmax(logits, dim=-1)[:, FAKE_CLASS_ID]
        semantic_labels = [0 if int(value) == REAL_CLASS_ID else 1 for value in class_ids]
        vit_test_true.extend(semantic_labels)
        vit_test_scores.extend(fake_probabilities.cpu().tolist())
print(f"Saved best validation-selected ViT and processor to {VIT_OUTPUT_DIR}")

## 6. Evaluate both fine-tuned models on the same held-out test images

In [ ]:
def binary_metrics(y_true, ai_scores):
    y_true = np.asarray(y_true, dtype=np.int64)
    ai_scores = np.asarray(ai_scores, dtype=np.float64)
    if len(y_true) != len(ai_scores) or not np.isfinite(ai_scores).all():
        raise ValueError("Invalid labels or model scores.")
    predicted = (ai_scores >= 0.5).astype(np.int64)
    return {
        "n": int(len(y_true)),
        "real_count": int(np.sum(y_true == 0)),
        "ai_generated_count": int(np.sum(y_true == 1)),
        "threshold": 0.5,
        "accuracy": float(accuracy_score(y_true, predicted)),
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "f1_score": float(f1_score(y_true, predicted, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, ai_scores)),
        "confusion_matrix_labels_0_1": confusion_matrix(y_true, predicted, labels=[0, 1]).tolist(),
    }

cnn_results = binary_metrics(cnn_true, cnn_scores)
vit_results = binary_metrics(vit_test_true, vit_test_scores)
if cnn_results["n"] != vit_results["n"] or cnn_results["n"] != len(test_frame):
    raise RuntimeError("Both models must be evaluated on the same complete test split.")

comparison = pd.DataFrame([
    {"model": "Fine-tuned CIFAKE Custom CNN", **cnn_results},
    {"model": "Fine-tuned Hugging Face ViT", **vit_results},
])
display(comparison[["model", "n", "accuracy", "precision", "recall", "f1_score", "roc_auc"]].style.format({
    "accuracy": "{:.2%}", "precision": "{:.2%}", "recall": "{:.2%}",
    "f1_score": "{:.2%}", "roc_auc": "{:.4f}"
}))

summary = {
    "dataset": "cashbowman/ai-generated-images-vs-real-images",
    "label_mapping": {"REAL": 0, "AI-generated": 1},
    "split_sizes": {
        "train": len(train_frame),
        "validation": len(validation_frame),
        "test": len(test_frame),
    },
    "starting_cnn_checkpoint": CNN_CHECKPOINT.name,
    "huggingface_model": MODEL_ID,
    "huggingface_revision": MODEL_REVISION,
    "results": {
        "cifake_custom_cnn": cnn_results,
        "huggingface_vit": vit_results,
    },
    "warning": (
        "Metrics apply only to this random held-out split from one dataset. The test "
        "split may contain images from the same sources or generators as training; "
        "evaluate on separate unseen sources and generators before claiming generalization."
    ),
}
(METRICS_DIR / "ai_vs_real_finetuning_comparison.json").write_text(
    json.dumps(summary, indent=2) + "\n", encoding="utf-8"
)
comparison.to_csv(METRICS_DIR / "ai_vs_real_finetuning_comparison.csv", index=False)
print(f"Metrics saved under {METRICS_DIR}")
print(f"CNN checkpoint: {CNN_OUTPUT_DIR / 'cifake_custom_cnn_ai_vs_real_finetuned.keras'}")
print(f"ViT model + processor: {VIT_OUTPUT_DIR}")

## 7. Export AI-vs-real checkpoints and metrics

The output directory contains the new CNN `.keras` model, the ViT model plus processor files, and JSON/CSV metrics. Downloading the ViT folder may be large; saving `OUTPUT_DIR` into mounted Google Drive is preferable for large files. The app will not switch to these checkpoints automatically.

In [ ]:
from google.colab import files
import shutil

OUTPUT_ARCHIVE = Path("/content/ai_vs_real_finetuning_outputs.zip")
if OUTPUT_ARCHIVE.exists():
    OUTPUT_ARCHIVE.unlink()
shutil.make_archive(str(OUTPUT_ARCHIVE.with_suffix("")), "zip", OUTPUT_DIR)
print(f"Created {OUTPUT_ARCHIVE} ({OUTPUT_ARCHIVE.stat().st_size / (1024**2):.1f} MiB)")
files.download(str(OUTPUT_ARCHIVE))